In [11]:
from pyspark.sql import SparkSession

spark=SparkSession.builder\
        .appName("QuickCommerce Streaming Pipeline") \
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.2") \
        .getOrCreate()

In [12]:
streaming_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers","localhost:9092") \
    .option("subscribe","ecommerce_topic") \
    .load()

In [17]:
checkpoint_dir ="C:/Users/scshi/Projects/RealTime-Data-Pipeline/tmp2/quickcommerce_streaming_checkpoint"

In [18]:
query = streaming_df.writeStream \
    .format("memory").queryName("realtime_ecommerce").outputMode("append").option("checkpointLocation", checkpoint_dir).start()
#query.awaitTermination()

In [ ]:
spark.sql('select * from realtime_ecommerce').show()

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]

: 

In [5]:
query.stop()

In [6]:
streaming_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
streaming_df.show()

AnalysisException: Queries with streaming sources must be executed with writeStream.start();
kafka

# TASK 2

In [8]:
streaming_df2= streaming_df.selectExpr("CAST(value AS STRING) as json") \
    .selectExpr("from_json(json, 'transaction_id STRING, user_id STRING, amount DOUBLE, timestamp STRING') AS data") \
    .select('data.*')